In [8]:
import json

json_path = r"C:/Users/PC/Desktop/chatbot/dataset/entry_test_questions_full.json"
with open(json_path, encoding="utf-8") as f:
    raw = json.load(f)

print(type(raw))
print("샘플 3개:", raw[:3])


<class 'dict'>


KeyError: slice(None, 3, None)

In [12]:
import os, json
from collections import Counter

REQ_KEYS = {"id","question","options","answer","image","level","category"}
VALID_LEVELS = {"초급","중급","고급"}

def validate(json_path: str, root_img: str, show=20):
    with open(json_path, encoding="utf-8") as f:
        data = json.load(f)

    items = data["items"] if isinstance(data, dict) and "items" in data else data
    errors, missing_imgs, ids = [], [], []

    for i, x in enumerate(items):
        miss = REQ_KEYS - set(x.keys())
        if miss:
            errors.append((i, x.get("id"), f"missing:{sorted(miss)}"))

        ids.append(x.get("id"))

        opts = x.get("options")
        if not isinstance(opts, list) or len(opts) < 2:
            errors.append((i, x.get("id"), "options invalid"))
        if x.get("answer") is None:
            errors.append((i, x.get("id"), "answer missing"))

        img_field = x.get("image", "")
        basename = os.path.basename(img_field) if isinstance(img_field, str) else ""
        img_path = os.path.join(root_img, basename)
        if not basename or not os.path.isfile(img_path):
            missing_imgs.append((i, x.get("id"), img_path))

    dup = [k for k, c in Counter(ids).items() if c > 1]

    print("# Duplicates:", dup)
    print("# Errors:", errors[:show], f"...({len(errors)})")
    print("# Missing images:", missing_imgs[:show], f"...({len(missing_imgs)})")
    return {"duplicates": dup, "errors": errors, "missing_imgs": missing_imgs}

# 👉 여기서 직접 실행
json_path = r"C:\Users\PC\Desktop\chatbot\dataset\entry_test_questions_full.json"
root_img = r"C:\Users\PC\Desktop\chatbot\dataset\entry_test_image"
validate(json_path, root_img, show=20)


# Duplicates: []
# Errors: [] ...(0)
# Missing images: [] ...(0)


{'duplicates': [], 'errors': [], 'missing_imgs': []}

In [ ]:
import pickle, json, sys
# 입력: 기존 pkl, 신규 문항 json, 출력 pkl
base_pkl, new_json, out_pkl = sys.argv[1:4]
emb = pickle.load(open(base_pkl,"rb"))
# emb: {id:{'embedding':np.array,'meta':{...}}}
data = json.load(open(new_json, encoding="utf-8"))
# TODO: KoBERT 임베더로 신규 질문/보기 텍스트 임베딩 생성해 emb에 추가
# save
pickle.dump(emb, open(out_pkl,"wb"))
print("saved:", out_pkl)
